# Cancer Genomics Pipeline - Quick Start

This notebook demonstrates how to run the complete cancer genomics analysis pipeline.

**Research Use Only**: This pipeline is for research purposes only and not for clinical or medical use.

## Setup

In [ ]:
import sys
sys.path.append('..')

from pipelines import CancerGenomicsPipeline
from models import DriverMutationClassifier, TreatmentResponsePredictor
from protein import ProteinStructurePredictor
from drug_discovery import MoleculeGenerator, MolecularDocking
from evaluation import PipelineEvaluator

print("Modules imported successfully!")

## 1. Run Complete Pipeline

In [ ]:
# Initialize pipeline
pipeline = CancerGenomicsPipeline(output_dir="../data/processed")

# Run analysis
results = pipeline.run_full_pipeline(
    tumor_data="../data/raw/tumor.bam",
    normal_data="../data/raw/normal.bam",
    hla_alleles=['HLA-A*02:01', 'HLA-B*07:02', 'HLA-C*07:01'],
    patient_id="example_patient"
)

## 2. View Results Summary

In [ ]:
print("Pipeline Results:")
print(f"  Total mutations: {results.get('annotated_mutations_count', 0)}")
print(f"  Driver mutations: {len(results.get('driver_mutations', []))}")
print(f"  Neoantigens: {len(results.get('neoantigens', []))}")
print(f"  Therapeutic targets: {len(results.get('therapeutic_targets', []))}")

## 3. Explore Driver Mutations

In [ ]:
import pandas as pd

driver_mutations = results.get('driver_mutations', pd.DataFrame())

if len(driver_mutations) > 0:
    print("Top Driver Mutations:")
    display(driver_mutations.head(10))
else:
    print("No driver mutations found")

## 4. Analyze Therapeutic Targets

In [ ]:
targets = results.get('therapeutic_targets', pd.DataFrame())

if len(targets) > 0:
    print("Therapeutic Targets:")
    display(targets)
    
    # Visualize target distribution
    import matplotlib.pyplot as plt
    
    target_types = targets['type'].value_counts()
    plt.figure(figsize=(8, 5))
    target_types.plot(kind='bar')
    plt.title('Therapeutic Target Types')
    plt.xlabel('Target Type')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No therapeutic targets identified")

## 5. Predict Treatment Response

In [ ]:
# Initialize treatment predictor
predictor = TreatmentResponsePredictor()

# Extract genomic signature
signature = predictor.extract_genomic_signature(driver_mutations)

print("Genomic Signature:")
print(f"  Mutation count: {signature['mutation_count']}")
print(f"  TMB: {signature['tumor_mutational_burden']:.1f} mut/Mb")
print(f"  Driver mutations: {', '.join(signature['driver_mutations'][:5])}")

# Predict drug responses
drugs = ['gefitinib', 'vemurafenib', 'pembrolizumab']

print("\nDrug Response Predictions:")
for drug in drugs:
    if drug == 'pembrolizumab':
        response = predictor.predict_immunotherapy_response(signature)
    else:
        response = predictor.predict_drug_response(signature, drug)
    
    print(f"\n{drug.upper()}:")
    print(f"  Response: {response.get('predicted_response', 'unknown')}")
    print(f"  Confidence: {response.get('confidence', 0):.2f}")

## 6. Evaluate Pipeline Performance

In [ ]:
# Evaluate pipeline
evaluator = PipelineEvaluator()
evaluation = evaluator.evaluate_pipeline(results)

print("Pipeline Evaluation:")
print(f"  Patient ID: {evaluation['pipeline_id']}")

if 'driver_prediction' in evaluation['metrics']:
    driver_eval = evaluation['metrics']['driver_prediction']
    print(f"\nDriver Prediction:")
    print(f"  Predicted drivers: {driver_eval['predicted_drivers']}")
    print(f"  True positives: {driver_eval['true_positives']}")
    print(f"  Precision: {driver_eval['precision']:.2f}")

if 'neoantigen_prediction' in evaluation['metrics']:
    neo_eval = evaluation['metrics']['neoantigen_prediction']
    print(f"\nNeoantigen Prediction:")
    print(f"  Total predictions: {neo_eval['total_predictions']}")
    print(f"  Strong binders: {neo_eval['strong_binders']}")

## Next Steps

- Explore individual notebooks for detailed analysis:
  - `mutation_analysis.ipynb` - Mutation visualization and analysis
  - `protein_structures.ipynb` - Protein structure prediction and analysis
  - `drug_discovery.ipynb` - Drug candidate generation and docking
- Review output files in `../data/processed/`
- Customize pipeline parameters for your specific use case

**Disclaimer**: This pipeline is for research purposes only and not for medical or clinical use.